In [1]:
#Post Training Quantization-Dynamic

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
X, y=make_moons(n_samples=5000,noise=0.5,random_state=42)

In [5]:
X

array([[ 0.64527536,  1.38251014],
       [ 0.14514823, -0.32157033],
       [ 0.11945131,  0.41631146],
       ...,
       [ 0.6360473 ,  0.66530771],
       [ 1.61542641, -0.24249711],
       [ 0.10599548,  1.0899585 ]], shape=(5000, 2))

In [6]:
X.shape

(5000, 2)

In [7]:
X=StandardScaler().fit_transform(X)

In [8]:
X=torch.tensor(X,dtype=torch.float32)

In [9]:
y=torch.tensor(y.reshape(-1,1),dtype=torch.float32)

In [10]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [26]:
class BigMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(2,128)
        self.fc2=nn.Linear(128,64)
        self.fc3=nn.Linear(64,64)
        self.fc4=nn.Linear(64,32)
        self.fc5=nn.Linear(32,16)
        self.fc6=nn.Linear(16,8)
        self.fc7=nn.Linear(8,1)
    
    def forward(self,x):
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=F.relu(self.fc3(x))
        x=F.relu(self.fc4(x))
        x=F.relu(self.fc5(x))
        x=F.relu(self.fc6(x))
        return torch.sigmoid(self.fc7(x))

In [27]:
model_fp32=BigMLP()

In [28]:
model_fp32.parameters()

<generator object Module.parameters at 0x0000018AC0C72500>

In [29]:
optimizer=torch.optim.Adam(model_fp32.parameters(),lr=0.01)

In [30]:
loss_fn=nn.BCELoss()

In [31]:
for epoch in range(2001):
    model_fp32.train()
    optimizer.zero_grad()
    out=model_fp32(X_train)
    loss=loss_fn(out,y_train)
    loss.backward()
    optimizer.step()
    if epoch %500==0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 0.6950
Epoch 500 | Loss: 0.3683
Epoch 1000 | Loss: 0.3264
Epoch 1500 | Loss: 0.2829
Epoch 2000 | Loss: 0.2524


In [32]:
def accuracy(model,X,y):
    model.eval()
    with torch.no_grad():
        preds=model(X)
        preds=(preds>0.5).float()
        return (preds==y).float().mean().item()
    

In [33]:
accuracy(model_fp32,X_test,y_test)

0.7940000295639038

In [34]:
#Quantization Techniques
from torch.quantization import quantize_dynamic

#dynamixc quantization
model_int8=quantize_dynamic(
    model_fp32,#model name
    {nn.Linear},#which layer in models to quantize
    dtype=torch.qint8
)


C:\Users\INMOR14\AppData\Local\Temp\ipykernel_21908\787055474.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8=quantize_dynamic(


In [35]:
print("INT8 Quantized accuracy:",accuracy(model_int8,X_test,y_test))

INT8 Quantized accuracy: 0.7590000033378601


In [36]:
import os
torch.save(model_fp32.state_dict(),"model_fp32.pt")
torch.save(model_int8.state_dict(),"model_dynamic_int8.pt")

In [ ]:
print("FP32 model size(MB): ",os.path.getsize("model_fp32.pt")/1e6)
print("INT8 model size(MB):  ",os.path.getsize("model_dynamic_int8.pt")/1e6)

FP32 model size(MB): 0.067317
INT8 model size(MB): 0.026373


In [25]:
#quantize manually

def quantize_tensor(t,num_bits=8):
    qmin=-2**(num_bits-1)
    qmax=2**(num_bits-1)-1
    rmin=t.min()
    rmax=t.max()
    scale=(rmax-rmin)/float(qmax-qmin+1e-8)
    zp=torch.round(-rmin/scale).to(torch.int8)
    q_t=torch.clamp(torch.round(t/scale)+zp,qmin,qmax).to(torch.int8)
    return q_t,scale,zp

In [26]:
def dequantize_tensor(q_t,scale,zp):
    return (q_t.float()-zp)*scale

In [40]:
new_model=BigMLP()

In [41]:
for name,param in new_model.named_parameters():
    print(name)
    print(param.shape)

fc1.weight
torch.Size([128, 2])
fc1.bias
torch.Size([128])
fc2.weight
torch.Size([64, 128])
fc2.bias
torch.Size([64])
fc3.weight
torch.Size([64, 64])
fc3.bias
torch.Size([64])
fc4.weight
torch.Size([32, 64])
fc4.bias
torch.Size([32])
fc5.weight
torch.Size([16, 32])
fc5.bias
torch.Size([16])
fc6.weight
torch.Size([8, 16])
fc6.bias
torch.Size([8])
fc7.weight
torch.Size([1, 8])
fc7.bias
torch.Size([1])


In [44]:
with torch.no_grad():
    for (name_fp,parm_fp),(name_q,param_q) in zip(model_fp32.named_parameters(),new_model.named_parameters()):
        q_param,scale,zp=quantize_tensor(parm_fp.data)
        dq_param=dequantize_tensor(q_param,scale,zp)
        param_q.data.copy_(dq_param)

In [45]:
print("INT8 Accuracy: ",accuracy(new_model,X_test,y_test))

INT8 Accuracy:  0.5130000114440918


In [3]:
X, y=make_moons(n_samples=500,noise=0.2,random_state=42)

In [4]:
X=torch.tensor(X,dtype=torch.float32)
y=torch.tensor(y.reshape(-1,1),dtype=torch.float32)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [19]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(2,16)
        self.fc2=nn.Linear(16,8)
        self.fc3=nn.Linear(8,1)

    def forward(self,x):
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))
    
model_fp32=MLP()

optimizer=torch.optim.AdamW(model_fp32.parameters(),lr=1e-4)

loss_fn=nn.BCELoss()

for epoch in range(2001):
    model_fp32.train()
    out=model_fp32(X_train)
    loss=loss_fn(out,y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

def accuracy(model,X,y):
    model.eval()
    with torch.no_grad():
        out=model(X)
        preds=(out>0.5).float()
        return (preds==y).float().mean().item()
    
print("FP32 Accuarcy: ",accuracy(model_fp32,X_test,y_test))



FP32 Accuarcy:  0.8600000143051147


In [ ]:
def get_activation_min_max(model,X_sample):
    act_ranges={}
    hooks=[]

    def register_hook(name):
        def hook_fn(module,input,output):
            tmin=output.min().item()
            tmax=output.max().item()
            act_ranges[name]=(tmin,tmax)
        return hook_fn

    for name,module in model.named_modules():
        if isinstance(module,nn.Linear):
            hook = module.register_forward_hook(register_hook(name))
            hooks.append(hook)

    model.eval()
    with torch.no_grad():  
        model(X_sample)

    for h in hooks:
        h.remove()

    return act_ranges

In [24]:
X_calib = X_train[:100]
     

act_ranges = get_activation_min_max(model_fp32, X_calib)
     

act_ranges
     

{'fc1': (-2.2548668384552, 1.9965834617614746),
 'fc2': (-0.7475477457046509, 2.6808173656463623),
 'fc3': (-1.9959733486175537, 3.634883165359497)}

In [39]:
class QuantizedMLP(nn.Module):
    def __init__(self,fp_model,act_ranges):
        super().__init__()
        self.fc1=nn.Linear(2,16)
        self.fc2=nn.Linear(16,8)
        self.fc3=nn.Linear(8,1)

        
        with torch.no_grad():
            for (name_fp,param_fp),(name_q,param_q) in zip(fp_model.named_parameters(),self.named_parameters()):
                quant_params,scale,zp=quantize_tensor(param_fp.data)
                dequantize_params=dequantize_tensor(quant_params,scale,zp)
                param_q.data.copy_(dequantize_params)
        
        self.act_scales={}
        for name,(amin,amax) in act_ranges.items():
            scale=(amax-amin)/255.0
            zp=round(-amin/scale) if scale>0 else 0
            self.act_scales[name]=(scale,zp)

        
    def quant_act(self,x,name):
        scale,zp=self.act_scales[name]
        q_t=torch.clamp(torch.round(x/scale)+zp,0,255).to(torch.uint8)
        d_t=(q_t.float()-zp)*scale
        return d_t
    
    def forward(self,x):
        x=F.relu(self.quant_act(self.fc1(x),'fc1'))
        x=F.relu(self.quant_act(self.fc2(x),'fc2'))
        return torch.sigmoid(self.fc3(x))



In [ ]:

model_static_ptq = QuantizedMLP(model_fp32, act_ranges)
print("STATIC PTQ Accuracy:", accuracy(model_static_ptq, X_test, y_test))

STATIC PTQ Accuracy: 0.7200000286102295
